# Lab 8-2: Prepare Concrete Quality Dataset using Spark MLlib Pipeline

This lab focuses on data exploration and preparation for Spark MLlib algorithms. The goal is to build a data engineering pipeline for the concrete quality dataset.

**Objectives**:
- Explore and verify schema and descriptive statistics
- Transform and encode features (e.g., age bucketing)
- Assemble features for modeling
- Standardize feature values
- Build and apply a Spark MLlib pipeline

**Files Used**:  
- `concrete_train.csv` (from [sparklab07data.zip](https://idsdl.csom.umn.edu/c/share/sparklab07data.zip))

**Topics**:  
- Data exploration (`describe`)
- Feature engineering (`SQLTransformer`, `Bucketizer`)
- Feature assembly (`VectorAssembler`)
- Feature scaling (`StandardScaler`)
- ML pipelines (`Pipeline`)

A description of fields is as follows. The `compressive_strength` is the target variable in this dataset.
|Field Name|Type|Description|
|--|--|--|
cement|Double|Cement content (kg/m³)|
blast_furnace_slag|Double|Blast furnace slag content (kg/m³)|
fly_ash|Double|Fly ash content (kg/m³)|
water|Double|Water content (kg/m³)|
superplasticizer|Double|Superplasticizer content (kg/m³)|
coarse_aggregate|Double|Coarse aggregate content (kg/m³)|
fine_aggregate|Double|Fine aggregate content (kg/m³)|
age|Double|Age of concrete (days)|
compressive_strength|Double|Concrete compressive strength (MPa)|

#### 1\. Download, unzip, and upload the data files needed for this lab.

The data file is part of this zip file [downloadable from here](https://idsdl.csom.umn.edu/c/share/sparklab07data.zip).

- first, create `catalog/schema/volume` as `msbabigdata/spark/lab8_2`
- manually upload the `concrete_train.csv` and `concrete_test.csv`

In [ ]:
%sql
create catalog if not exists msbabigdata;
create schema if not exists msbabigdata.spark;
create volume if not exists msbabigdata.spark.lab8_2;

#### 2\. File format inspection

- Display the first 500 bytes of `concrete_train.csv` and `concrete_test.csv` using DBFS utils API. 

In [ ]:
path = "/Volumes/msbabigdata/spark/lab8_2"
print(dbutils.fs.head(path+"/concrete_train.csv", 500))

[Truncated to first 500 bytes]
540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
266.0,114.0,0.0,228.0,0.0,932.0,670.0,90,47.03
380.0,95.0,0.0,228.0,0.0,932.0,594.0,365,43.70
380.0,95.0,0.0,228.0,0.0,932.0,594.0,28,36.45
266.0,114.0,0.0,228.0,0.0,932.0,670.0,28,45.85
475.0,0.0,0.0,228.0,0.0,932.0,594.0,28,39.29
198.6,132.4,0.0,192.0,


In [ ]:
print(dbutils.fs.head(path+"/concrete_test.csv", 500))

[Truncated to first 500 bytes]
150.0,237.0,0.0,174.0,12.0,1069.0,675.0,28,37.43
144.0,170.0,133.0,192.0,8.0,814.0,805.0,28,29.87
331.0,170.0,0.0,195.0,8.0,811.0,802.0,28,56.61
155.0,0.0,143.0,193.0,9.0,1047.0,697.0,28,12.46
155.0,183.0,0.0,193.0,9.0,877.0,868.0,28,23.79
135.0,0.0,166.0,180.0,10.0,961.0,805.0,28,13.29
266.0,112.0,87.0,178.0,10.0,910.0,745.0,28,39.42
314.0,145.0,113.0,179.0,8.0,869.0,690.0,28,46.23
313.0,145.0,0.0,127.0,8.0,1000.0,822.0,28,44.52
146.0,173.0,0.0,182.0,3.0,986.0,817.0,28,23.74
144.0,136


#### 3\. Load the `concrete_train.csv` file into a dataframe `train`. 

- then verify its schema and content

In [ ]:
schema = "cement double, blast_furnace_slag double, fly_ash double, water double, superplasticizer double, course_aggregate double, fine_aggregate double, age int, compressive_strength double"

In [ ]:
train = spark.read.csv(path+"/concrete_train.csv", schema=schema, header = False)
train.limit(10).display()

cement | blast_furnace_slag | fly_ash | water | superplasticizer | course_aggregate | fine_aggregate | age | compressive_strength
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1040.0 | 676.0 | 28 | 79.99
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1055.0 | 676.0 | 28 | 61.89
332.5 | 142.5 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 270 | 40.27
332.5 | 142.5 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 365 | 41.05
198.6 | 132.4 | 0.0 | 192.0 | 0.0 | 978.4 | 825.5 | 360 | 44.3
266.0 | 114.0 | 0.0 | 228.0 | 0.0 | 932.0 | 670.0 | 90 | 47.03
380.0 | 95.0 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 365 | 43.7
380.0 | 95.0 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 28 | 36.45
266.0 | 114.0 | 0.0 | 228.0 | 0.0 | 932.0 | 670.0 | 28 | 45.85
475.0 | 0.0 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 28 | 39.29

#### 4\. Obtain descriptive statistics.

Because this dataset has numerical columns, it is useful to obtain descriptive statistics (using `.describe()` DataFrame API). Report the descriptive statistics

In [ ]:
train.describe().display()

summary | cement | blast_furnace_slag | fly_ash | water | superplasticizer | course_aggregate | fine_aggregate | age | compressive_strength
count | 900 | 900 | 900 | 900 | 900 | 900 | 900 | 900 | 900
mean | 290.563222222222 | 68.4981111111111 | 49.03222222222221 | 180.72266666666667 | 5.810888888888885 | 981.0683333333328 | 776.2333333333324 | 48.21333333333333 | 36.37484444444442
stddev | 104.64460120499346 | 85.81753820138701 | 61.898696038381885 | 21.761715870809013 | 6.173544447428135 | 75.5029483066941 | 81.38528207195112 | 67.20007349431869 | 17.210366992350004
min | 102.0 | 0.0 | 0.0 | 121.8 | 0.0 | 801.0 | 594.0 | 1 | 2.33
max | 540.0 | 359.4 | 200.0 | 247.0 | 32.2 | 1145.0 | 992.6 | 365 | 82.6

#### 5\. Convert  the Age Field 

[The SQLTransformer()](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.SQLTransformer.html) provides a way to transform a DataFrame using a SQL query statement. Currently it only supports SQL syntax like `"SELECT … FROM __THIS__"` where `__THIS__` represents the table for the focal DataFrame.

Define an SQLTransformer `st` that creates a new field `age_enc` that takes the following values:

- 0, if age < 30
- 1, if age is >= 30 and < 90
- 2, if age is >= 90 and < 180
- 3, if age is 180 or above

Alternatively, you may use the specialized feature transformer [bucketizer](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.Bucketizer.html).
```python
from pyspark.ml.feature import Bucketizer
bt = Bucketizer(inputCol="age",splits=[-float("inf"),30,90,180,float("inf")], outputCol="age_bt")
train_bt = bt.transform(train_st)
```

In [ ]:
from pyspark.ml.feature import SQLTransformer
st = SQLTransformer(
                    statement="""
                    SELECT *, case when (age < 30) then 0
                    when (age >= 30 and age < 60) then 1
                    else 2 end as age_enc
                    FROM __THIS__
                    """)
df_st = st.transform(train)
df_st.limit(5).display()


cement | blast_furnace_slag | fly_ash | water | superplasticizer | course_aggregate | fine_aggregate | age | compressive_strength | age_enc
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1040.0 | 676.0 | 28 | 79.99 | 0
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1055.0 | 676.0 | 28 | 61.89 | 0
332.5 | 142.5 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 270 | 40.27 | 2
332.5 | 142.5 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 365 | 41.05 | 2
198.6 | 132.4 | 0.0 | 192.0 | 0.0 | 978.4 | 825.5 | 360 | 44.3 | 2

#### 6\. Assemble the features column

Use [a `VectorAssembler`](https://spark.apache.org/docs/latest/ml-features.html#vectorassembler) to create a new `features` column with all fields except `compressive_strength` and `age`. 


In [ ]:
train.columns

In [ ]:
from pyspark.ml.feature import VectorAssembler

featureCols = ['cement',
 'blast_furnace_slag',
 'fly_ash',
 'water',
 'superplasticizer',
 'course_aggregate',
 'fine_aggregate',
 'age_enc']

va = VectorAssembler(inputCols=featureCols, outputCol="features")

 #test it
df_va = va.transform(df_st)
df_va.limit(5).display()
 

cement | blast_furnace_slag | fly_ash | water | superplasticizer | course_aggregate | fine_aggregate | age | compressive_strength | age_enc | features
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1040.0 | 676.0 | 28 | 79.99 | 0 | {"type":"1","size":null,"indices":null,"values":["540.0","0.0","0.0","162.0","2.5","1040.0","676.0","0.0"]}
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1055.0 | 676.0 | 28 | 61.89 | 0 | {"type":"1","size":null,"indices":null,"values":["540.0","0.0","0.0","162.0","2.5","1055.0","676.0","0.0"]}
332.5 | 142.5 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 270 | 40.27 | 2 | {"type":"1","size":null,"indices":null,"values":["332.5","142.5","0.0","228.0","0.0","932.0","594.0","2.0"]}
332.5 | 142.5 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 365 | 41.05 | 2 | {"type":"1","size":null,"indices":null,"values":["332.5","142.5","0.0","228.0","0.0","932.0","594.0","2.0"]}
198.6 | 132.4 | 0.0 | 192.0 | 0.0 | 978.4 | 825.5 | 360 | 44.3 | 2 | {"type":"1","size":null,"indices":null,"values":["198.6","132.4","0.0"

Verify your assembler does what it needs to do:

#### 7\. Standardize feature values.

[The estimator `StandardScaler`](https://spark.apache.org/docs/latest/ml-features.html#standardscaler) transforms a dataset of Vector rows, normalizing each feature to have unit standard deviation and/or zero mean. It takes parameters:

- `withStd`: True by default. Scales the data to unit standard deviation.
- `withMean`: False by default. Centers the data with mean before scaling. It will lead to many non-zero elements, so take care when applying this to sparse data.

Create an instance of `StandardScaler` called `ss`

- it should apply to `features` and create a new column `scaledfeatures`

In [ ]:
from pyspark.ml.feature import StandardScaler
ss = StandardScaler(inputCol="features", outputCol="scaled_features")

# test it

df_ss = ss.fit(df_va).transform(df_va)
df_ss.limit(5).display()


cement | blast_furnace_slag | fly_ash | water | superplasticizer | course_aggregate | fine_aggregate | age | compressive_strength | age_enc | features | scaled_features
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1040.0 | 676.0 | 28 | 79.99 | 0 | {"type":"1","size":null,"indices":null,"values":["540.0","0.0","0.0","162.0","2.5","1040.0","676.0","0.0"]} | {"type":"1","size":null,"indices":null,"values":["5.160323550205591","0.0","0.0","7.444265928373107","0.4049537540855483","13.77429654502367","8.30617014268455","0.0"]}
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1055.0 | 676.0 | 28 | 61.89 | 0 | {"type":"1","size":null,"indices":null,"values":["540.0","0.0","0.0","162.0","2.5","1055.0","676.0","0.0"]} | {"type":"1","size":null,"indices":null,"values":["5.160323550205591","0.0","0.0","7.444265928373107","0.4049537540855483","13.97296428365382","8.30617014268455","0.0"]}
332.5 | 142.5 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 270 | 40.27 | 2 | {"type":"1","size":null,"indices":null,"values":["332.5","142.5","0

Verify your standard scaler

#### 8\. Create a pipeline

Create an instance of [an estimator `Pipeline`](https://spark.apache.org/docs/latest/ml-pipeline.html#example-pipeline) called `pl`

-  Its `stages` should include the SQLTransformer, VectorAssembler and StandardScaler.

In [ ]:
from pyspark.ml import Pipeline

pl = Pipeline(stages = [st, va,ss])



#### 9\. Train the `Pipeline` on the training and use the trained pipeline.

- First train a pipeline model `pl_model` on the training data.
- use the `pl_model` to prepare both the training dataset and the test data (load it to a `test` DataFrame first), making sure that it works
- inspect the results

In [ ]:
pl_model = pl.fit(train)

In [ ]:
train_pl = pl_model.transform(train)
train_pl.limit(5).display()

cement | blast_furnace_slag | fly_ash | water | superplasticizer | course_aggregate | fine_aggregate | age | compressive_strength | age_enc | features | scaled_features
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1040.0 | 676.0 | 28 | 79.99 | 0 | {"type":"1","size":null,"indices":null,"values":["540.0","0.0","0.0","162.0","2.5","1040.0","676.0","0.0"]} | {"type":"1","size":null,"indices":null,"values":["5.160323550205591","0.0","0.0","7.444265928373107","0.4049537540855483","13.77429654502367","8.30617014268455","0.0"]}
540.0 | 0.0 | 0.0 | 162.0 | 2.5 | 1055.0 | 676.0 | 28 | 61.89 | 0 | {"type":"1","size":null,"indices":null,"values":["540.0","0.0","0.0","162.0","2.5","1055.0","676.0","0.0"]} | {"type":"1","size":null,"indices":null,"values":["5.160323550205591","0.0","0.0","7.444265928373107","0.4049537540855483","13.97296428365382","8.30617014268455","0.0"]}
332.5 | 142.5 | 0.0 | 228.0 | 0.0 | 932.0 | 594.0 | 270 | 40.27 | 2 | {"type":"1","size":null,"indices":null,"values":["332.5","142.5","0

Inspect `scaledFeatures` and `features` column in the transformed dataset

This is the end of the lab.